<div align="center">

# NOKTA Workshop: AI-Based Environmental Data Analysis and Interpretation

**Colab notebook of the two-hour workshop** · NOKTA, Objective Measurement, Participation, Monitoring and Action Network · ÜNİDES 2026

**Prof. Dr. Utku Köse**<br>
Süleyman Demirel University, Isparta, Türkiye · AI Application and Research Center (YAZEM)<br>
University of North Dakota · Universidad Panamericana · Vel Tech University

[Workshop site](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/) · [GitHub](https://github.com/utkukose/env-data-ai-nokta2026-NB-workshop) · [ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com)

*This workshop is updated in line with current developments in the field. First version: November 2026.*

</div>

## How to use this notebook

Save a copy to your Drive first (File, Save a copy in Drive). Run the cells in order with Shift+Enter. Each section belongs to one part of the [workshop site](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/): Section 1 to Part 1, up to Section 5 for Part 5. No installation is needed on Colab.

- **Python note** cells explain a piece of the Python language where it first appears.
- **Exercises** give instant feedback with `check(...)`. Reference solutions sit in folded cells and should be opened only after trying.
- **Observe and reflect** asks open questions about an output and has no solution cell.
- The data are synthetic and follow the NOKTA protocol. Section 0 shows how to load real NOKTA files instead.

## 0. Setup and data

The first cell loads the libraries and the helper `check`. The second cell reads four files: The readings (one row every 10 seconds), the field forms (one row per visit), the side-by-side test of the three stations and, for the synthetic data only, a list of the problems that were planted on purpose.

**Your own NOKTA data.** Set `DATA_SOURCE = "own"` and upload your CSV files with the columns of `data/nokta_template.csv` (Files panel on the left). If your logger uses other column names, fill `COLUMN_MAP`. Everything after this cell then runs on your data.

In [ ]:
import json, re, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RANDOM_STATE = 2026
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:                      # matplotlib is used instead
    HAS_PLOTLY = False

TYPE_ORDER = ["shaded", "traffic", "reference", "sunny_hard", "crowded", "green"]
TYPE_COLOR = {"shaded": "#2a78d6", "traffic": "#eb6834", "reference": "#1baf7a",
              "sunny_hard": "#eda100", "crowded": "#e87ba4", "green": "#008300"}
TYPE_NAME = {"shaded": "Shaded", "traffic": "Traffic",
             "reference": "Reference", "sunny_hard": "Sunny hard",
             "crowded": "Busy", "green": "Green"}


def check(name, value, tol=0.01):
    """Compare an exercise answer with the expected answer."""
    exp = EXPECTED.get(name)
    if value is None:
        print(f"{name}: no answer yet")
        return False
    if isinstance(exp, str):
        ok = str(value).strip().lower() == exp
    else:
        ok = abs(float(value) - exp) <= tol
    print(f"{name}: " + ("correct ✓" if ok else "not yet, try again"))
    return ok

print("Ready. Plotly for 3D plots:", HAS_PLOTLY)

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
EXPECTED = {
    'Exercise 1.A': 139.4529,
    'Exercise 2.A': -1.105,
    'Exercise 2.B': 9.2611,
    'Exercise 3.A': 5,
    'Exercise 4.A': 'vehicles_per_min',
    'Exercise 5.A': 4,
}

In [ ]:
DATA_SOURCE = "synthetic"        # "synthetic" or "own"
REPO_RAW = ("https://raw.githubusercontent.com/utkukose/"
            "env-data-ai-nokta2026-NB-workshop/main/data/")
FILES = {"readings": "nokta_synthetic.csv",
         "forms": "nokta_field_forms_synthetic.csv",
         "coloc": "nokta_colocation_synthetic.csv",
         "truth": "nokta_synthetic_qc_truth.csv"}
OWN_FILES = {"readings": "my_readings.csv", "forms": "my_field_forms.csv",
             "coloc": "my_colocation.csv"}
COLUMN_MAP = {}                    # e.g. {"Temp": "temperature_c"}


def fetch(name):
    """Local copy if present, otherwise a download from GitHub."""
    for folder in (Path("data"), Path("../data"), Path("workshop_data")):
        if (folder / name).exists():
            return folder / name
    Path("workshop_data").mkdir(exist_ok=True)
    target = Path("workshop_data") / name
    urllib.request.urlretrieve(REPO_RAW + name, target)
    return target


if DATA_SOURCE == "own":
    paths = {k: Path(v) for k, v in OWN_FILES.items()}
else:
    paths = {k: fetch(v) for k, v in FILES.items()}
print({k: str(v) for k, v in paths.items()})

## Python step: Five ideas for reading the code

Participants without Python can read this step in five minutes and skip it in the live session.

1. **Variables** keep a value under a name: `rh = 43.0`.
2. **Lists** keep several values in order: `pm = [14.2, 15.0, 13.8]`, and `pm[0]` is the first one.
3. **Dictionaries** link names to values: `{"P1": "shaded"}`.
4. **Functions** package a calculation: `def dew_point(t, rh): ...` is run with `dew_point(13.3, 43)`.
5. **pandas tables (DataFrames)** hold the NOKTA readings: One row per reading, one column per channel. `df["pm25_ugm3"]` is one column, `df.groupby("point_type")` splits the rows by point.

In [ ]:
# Python practice: a small example for learning the language
pm = [14.2, 15.0, 13.8, 14.6, 97.3, 15.1, 14.4]      # a list of PM2.5 readings
print("number of readings:", len(pm))
print("mean:", round(sum(pm) / len(pm), 2))
print("median:", sorted(pm)[len(pm) // 2])
point = {"id": "P2", "type": "traffic", "vehicles_15min": 104}
print(f"{point['id']}: {point['type']},",
      point["vehicles_15min"], "vehicles in 15 minutes")

**Python note.** `f"..."` is a formatted text: Values inside `{ }` are written into it. `round(x, 2)` keeps two decimals, `sorted(...)` returns a sorted copy and `//` divides without the remainder.

## 1. Measure: The campus as data

*Workshop site: [Part 1](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/part1.html)*

### 1.1 Load and look

In [ ]:
def load_readings(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df["visit"] = (df["campus"] + "-R" + df["round"].astype(str) + "-"
                   + df["point_id"])
    return df


CHANNELS = ['temperature_c', 'rel_humidity_pct', 'pressure_hpa', 'pm1_ugm3',
    'pm25_ugm3', 'pm10_ugm3', 'co2_ppm', 'light_lux', 'uva_wm2', 'uvb_wm2']
PM = ['pm1_ugm3', 'pm25_ugm3', 'pm10_ugm3']
LUX_MAX = 65535

df = load_readings(paths["readings"])
df = df.rename(columns=COLUMN_MAP)
forms = pd.read_csv(paths["forms"])
coloc = pd.read_csv(paths["coloc"])
print(df.shape[0], "readings,", forms.shape[0], "visits,",
      df["campus"].nunique(), "campuses")
display(df.head(3))
display(df.groupby(["campus", "round"])["point_id"].nunique().unstack())

**Python note.** `df.groupby([...])[col].nunique()` counts the different values of a column in each group, and `.unstack()` turns the inner group into columns.

In [ ]:
visit = df[df["visit"] == "A-R1-P2"]     # Campus A, Round 1, near traffic
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, ch in zip(axes, ["temperature_c", "pm25_ugm3", "co2_ppm"]):
    ax.plot(visit["seconds_since_arrival"] / 60, visit[ch], lw=1.4, color="#0E7A5F")
    ax.axvspan(0, 2, color="#eda100", alpha=0.15)
    ax.set_title(ch); ax.set_xlabel("minutes since arrival")
plt.tight_layout(); plt.show()

### 1.2 Quality flags

Raw data are never changed. The functions below add one true or false column per problem: Warm-up after a move, logger gaps, a light sensor at its limit, humidity above 85 %, single-channel glitches, CO₂ out of range and a frozen CO₂ sensor. The spike test uses the modified z-score of Part 1:

$$M_i = \frac{0.6745\,(x_i - \tilde{x})}{\mathrm{MAD}}$$

*From the formulas to the code: `robust_z` is M, `spike_scores` applies it to the distance from a running median of five readings.*

In [ ]:
def robust_z(x):
    """Modified z-score: 0.6745 (x - median) / MAD (Leys et al., 2013)."""
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad == 0:
        return np.zeros_like(x)
    return 0.6745 * (x - med) / mad

def spike_scores(series, window=5):
    """Robust z-score of the distance from a running median."""
    run_med = series.rolling(window, center=True, min_periods=1).median()
    return pd.Series(robust_z(series - run_med), index=series.index)

def flag_quality(df, z_limit=6.0, warmup_s=120, flat_len=12):
    """Return one boolean column per quality problem (raw data are kept)."""
    f = pd.DataFrame(index=df.index)
    f["warmup"] = df["seconds_since_arrival"] < warmup_s
    f["gap"] = df[CHANNELS].isna().any(axis=1)
    f["saturated_light"] = df["light_lux"] >= LUX_MAX
    f["high_rh"] = df["rel_humidity_pct"] > 85
    jump = {}
    for ch in PM:
        z = df.groupby("visit")[ch].transform(spike_scores)
        run_med = df.groupby("visit")[ch].transform(
            lambda s: s.rolling(5, center=True, min_periods=1).median())
        jump[ch] = (z.abs() > z_limit) & (df[ch] > 2 * run_med)
    jump = pd.DataFrame(jump)
    for ch in PM:
        # a real plume lifts every particle size, a glitch lifts only one
        f["spike_" + ch] = jump[ch] & ~jump.drop(columns=ch).any(axis=1)
    f["spike_co2_ppm"] = (df["co2_ppm"] <= 300) | (df["co2_ppm"] >= 2000)
    roll_sd = df.groupby("visit")["co2_ppm"].transform(
        lambda s: s.rolling(flat_len, min_periods=flat_len).std())
    end_of_run = (roll_sd == 0).astype(int)
    # spread the flag back over the whole frozen window
    spread = end_of_run[::-1].groupby(df["visit"][::-1]).transform(
        lambda s: s.rolling(flat_len, min_periods=1).max())[::-1]
    f["flatline_co2"] = spread.fillna(0).astype(bool)
    return f

def clean_readings(df, flags):
    """Copy of the data with flagged values set to missing (never to zero)."""
    out = df.copy()
    out.loc[flags["warmup"], CHANNELS] = np.nan
    out.loc[flags["saturated_light"], "light_lux"] = np.nan
    out.loc[flags["flatline_co2"] | flags["spike_co2_ppm"], "co2_ppm"] = np.nan
    for ch in PM:
        out.loc[flags["spike_" + ch], ch] = np.nan
    return out


flags = flag_quality(df)
display(flags.sum().rename("flagged readings").to_frame().T)

In [ ]:
if "truth" in paths:     # only the synthetic data have planted problems
    truth = pd.read_csv(paths["truth"])["qc_truth"].to_numpy()
    pairs = {"warmup": "warmup", "spike_pm25": "spike_pm25_ugm3",
             "spike_pm10": "spike_pm10_ugm3", "spike_co2": "spike_co2_ppm",
             "flatline_co2": "flatline_co2", "gap": "gap"}
    rows = [dict(problem=k, planted=int((truth == k).sum()),
                 flagged=int(flags[v].sum()),
                 caught=int(((truth == k) & flags[v].to_numpy()).sum()))
            for k, v in pairs.items()]
    display(pd.DataFrame(rows))
v = df["visit"] == "B-R1-P1"
plt.figure(figsize=(9, 3))
plt.plot(df.loc[v, "seconds_since_arrival"] / 60, df.loc[v, "co2_ppm"],
         color="#0E7A5F")
bad = v & (flags["flatline_co2"] | flags["spike_co2_ppm"])
plt.scatter(df.loc[bad, "seconds_since_arrival"] / 60, df.loc[bad, "co2_ppm"],
            facecolors="none", edgecolors="#B3261E", s=50, label="flagged")
plt.axvspan(0, 2, color="#eda100", alpha=0.15, label="warm-up")
plt.ylabel("CO₂ (ppm)"); plt.xlabel("minutes since arrival")
plt.legend(); plt.show()

### Exercise 1.A

Compute the modified z-score of the reading 97.3 among the seven readings of the worked example with `robust_z`. Store the value in `answer`.

In [ ]:
readings = np.array([14.2, 15.0, 13.8, 14.6, 97.3, 15.1, 14.4])
answer = None   # your answer
check("Exercise 1.A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = robust_z(readings)[4]
print(round(answer, 2))
check("Exercise 1.A", answer)

### 1.3 A second opinion from AI: Isolation Forest

Rules catch the problems that someone thought of. An Isolation Forest looks for readings that are easy to separate from the rest in all channels at once [Liu et al., 2008]. Its flags are a list to review with the field notes, not a list to delete.

In [ ]:
from sklearn.ensemble import IsolationForest

X_if = df[CHANNELS].copy()
X_if = (X_if - X_if.groupby(df["visit"]).transform("median")).fillna(0)
iso = IsolationForest(n_estimators=200, contamination=0.02,
                      random_state=RANDOM_STATE).fit(X_if)
df["if_flag"] = iso.predict(X_if) == -1
rule_any = flags.drop(columns=["high_rh"]).any(axis=1)
print("Isolation Forest flags:", int(df["if_flag"].sum()))
print("of these, also flagged by a rule:",
      int((df["if_flag"] & rule_any).sum()))
display(df.loc[df["if_flag"] & ~rule_any,
               ["visit", "seconds_since_arrival", "pm25_ugm3", "co2_ppm"]].head(8))

**Missing is not zero.** One logger restart left six empty readings. Compare the mean when they are skipped with the mean when they are written as zero.

In [ ]:
gap = df[df["visit"] == "A-R2-P5"]["temperature_c"].iloc[30:52]
print("skipped:", round(gap.mean(), 2), "°C   written as zero:",
      round(gap.fillna(0).mean(), 2), "°C")

### 1.4 Three stations, one story

$$o_s = \frac{1}{T}\sum_{t=1}^{T}\bigl(x_s(t) - \bar{x}(t)\bigr)$$

*From the formulas to the code: `colocation_offsets` computes o for temperature, humidity and CO₂ and a ratio for particles. `apply_offsets` then subtracts or divides.*

In [ ]:
def colocation_offsets(coloc):
    """Offset of each station from the mean of the three stations."""
    wide = {ch: coloc.pivot(index="timestamp", columns="station", values=ch)
            for ch in ["temperature_c", "rel_humidity_pct", "pm25_ugm3", "co2_ppm"]}
    rows = []
    for s in sorted(coloc["station"].unique()):
        rows.append(dict(
            station=s,
            temperature_c=(wide["temperature_c"][s]
                           - wide["temperature_c"].mean(axis=1)).mean(),
            rel_humidity_pct=(wide["rel_humidity_pct"][s]
                              - wide["rel_humidity_pct"].mean(axis=1)).mean(),
            co2_ppm=(wide["co2_ppm"][s] - wide["co2_ppm"].mean(axis=1)).mean(),
            pm_ratio=(wide["pm25_ugm3"][s] / wide["pm25_ugm3"].mean(axis=1)).mean(),
        ))
    return pd.DataFrame(rows).set_index("station")

def apply_offsets(df, offsets):
    out = df.copy()
    st = out["station"]
    for ch in ["temperature_c", "rel_humidity_pct", "co2_ppm"]:
        out[ch] = out[ch] - st.map(offsets[ch])
    for ch in PM:
        out[ch] = out[ch] / st.map(offsets["pm_ratio"])
    return out


offsets = colocation_offsets(coloc)
display(offsets.round(3))

### 1.5 Humid air and the particle counter

$$C(a_w) = 1 + \frac{\kappa\,\rho_w/\rho_p}{-1 + 1/a_w}, \qquad a_w = \mathrm{RH}/100$$

The correction of Crilley et al. (2018) uses κ = 0.3 and ρ<sub>p</sub> = 1.65 g/cm³. A real study estimates κ for the local air against a reference instrument.

In [ ]:
def humidity_factor(rh, kappa=0.3, rho_p=1.65):
    """Growth factor C of particle mass with humidity (Crilley et al., 2018)."""
    aw = np.clip(np.asarray(rh, dtype=float), 1, 99) / 100.0
    return 1.0 + (kappa / rho_p) / (-1.0 + 1.0 / aw)

def correct_pm(df):
    out = df.copy()
    c = humidity_factor(out["rel_humidity_pct"])
    for ch in PM:
        out[ch] = out[ch] / c
    return out


for rh in (50, 70, 85, 90):
    print(f"RH {rh} %  ->  C = {humidity_factor(rh):.2f}")

### 1.6 The table of clean minutes

Flagged values become missing, the station offsets are removed, particles are corrected for humidity and every visit is summarized minute by minute. Parts 2 to 4 start from this table.

In [ ]:
def minute_table(df):
    """Median of every channel for each minute of each visit."""
    d = df.copy()
    d["minute"] = d["seconds_since_arrival"] // 60
    keys = ["campus", "round", "point_id", "point_type", "visit_order",
            "station", "visit", "minute"]
    m = d.groupby(keys)[CHANNELS].median().reset_index()
    t0 = d.groupby(keys)["timestamp"].min().reset_index(drop=True)
    m["hour"] = t0.dt.hour + t0.dt.minute / 60.0
    return m.dropna(subset=["temperature_c"]).reset_index(drop=True)


clean = clean_readings(df, flags)
adjusted_raw_pm = apply_offsets(clean, offsets)
adjusted = correct_pm(adjusted_raw_pm)
minutes = minute_table(adjusted)
minutes_raw_pm = minute_table(adjusted_raw_pm)
minutes.to_csv("nokta_clean_minutes.csv", index=False)
print(len(minutes), "clean minutes saved to nokta_clean_minutes.csv")
c2 = (minutes.campus == "C") & (minutes["round"] == 2)
display(pd.DataFrame({
    "raw PM2.5": minutes_raw_pm[c2].groupby("point_type")["pm25_ugm3"].median(),
    "corrected": minutes[c2].groupby("point_type")["pm25_ugm3"].median(),
    "RH %": minutes[c2].groupby("point_type")["rel_humidity_pct"].median(),
}).round(2))

**Observe and reflect.** Which of the flags would you check first with the field notes, and why? Which problems could a rule never find, but a team member at the point would notice at once?

## 2. Analyze: Fair comparisons

*Workshop site: [Part 2](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/part2.html)*

### 2.1 The clock behind every point

In [ ]:
def visit_table(minutes):
    keys = ["campus", "round", "point_id", "point_type", "visit_order"]
    v = minutes.groupby(keys)[CHANNELS + ["hour"]].median().reset_index()
    return v


visits = visit_table(minutes)
a = visits[visits.campus == "A"]
fig, ax = plt.subplots(figsize=(8, 3.6))
for rd, style in ((1, "-"), (2, "--")):
    r = a[a["round"] == rd].sort_values("hour")
    ax.plot(r["hour"], r["temperature_c"], style, color="gray", lw=1)
    colors = [TYPE_COLOR[t] for t in r.point_type]
    ax.scatter(r["hour"], r["temperature_c"], c=colors, s=70, zorder=3)
ax.set_xlabel("clock time (hours)"); ax.set_ylabel("°C")
ax.set_title("Campus A: Round 1 (solid) and Round 2 (dashed)")
plt.show()

### 2.2 The two-round mean and the reference point

$$\bar{x}_p = \frac{x_{p,1} + x_{p,2}}{2}, \qquad \Delta_p = \bar{x}_p - \bar{x}_{\text{ref}}$$

In [ ]:
def fair_table(visits, channel):
    """Round 1, Round 2, their mean and the difference from the reference."""
    w = visits.pivot_table(index=["campus", "point_type"], columns="round",
                           values=channel).rename(columns={1: "round1", 2: "round2"})
    w["two_round_mean"] = (w["round1"] + w["round2"]) / 2
    ref = w.xs("reference", level="point_type")["two_round_mean"]
    w["minus_reference"] = w["two_round_mean"] - ref.reindex(
        w.index.get_level_values("campus")).values
    return w.reset_index()


fair_t = fair_table(visits, "temperature_c")
display(fair_t[fair_t.campus == "A"].round(2))
fair_pm = fair_table(visits, "pm25_ugm3")
pivot = fair_pm.pivot(index="point_type", columns="campus", values="minus_reference")
pivot.loc[TYPE_ORDER].plot.barh(figsize=(7, 3.6),
                                color=["#2a78d6", "#eb6834", "#1baf7a"])
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("PM2.5 minus the reference point (µg/m³)")
plt.show()

### Exercise 2.A

Using `fair_t`, find the difference of the green area from the reference point on Campus A (°C).

In [ ]:
answer = None   # your answer
check("Exercise 2.A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
row = fair_t[(fair_t.campus == "A") & (fair_t.point_type == "green")]
answer = row["minus_reference"].iloc[0]
print(round(answer, 2))
check("Exercise 2.A", answer)

### 2.3 Typical values

In [ ]:
def add_anomalies(minutes):
    m = minutes.copy()
    grp = m.groupby(["campus", "round"])
    m["temp_anomaly"] = m["temperature_c"] - grp["temperature_c"].transform("median")
    m["pm25_anomaly"] = m["pm25_ugm3"] - grp["pm25_ugm3"].transform("median")
    m["co2_anomaly"] = m["co2_ppm"] - grp["co2_ppm"].transform("median")
    m["log10_lux"] = np.log10(m["light_lux"].fillna(LUX_MAX))
    return m


anom = add_anomalies(minutes)
fig, ax = plt.subplots(figsize=(9, 3.6))
data = [anom.loc[anom.point_type == t, "pm25_anomaly"].dropna() for t in TYPE_ORDER]
bp = ax.boxplot(data, patch_artist=True, showmeans=True)
for patch, t in zip(bp["boxes"], TYPE_ORDER):
    patch.set_facecolor(TYPE_COLOR[t])
ax.set_xticks(range(1, 7), [TYPE_NAME[t] for t in TYPE_ORDER])
ax.set_ylabel("PM2.5 minus campus-round median (µg/m³)")
plt.show()
display(anom.groupby("point_type")["pm25_anomaly"].agg(["median", "mean"]).round(2))

### 2.4 Related or caused?

In [ ]:
def pearson_r(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    return float(np.corrcoef(x[ok], y[ok])[0, 1])


r_all = pearson_r(minutes.temperature_c, minutes.pressure_hpa)
print("all minutes:", round(r_all, 3))
for (c, rd), g in minutes.groupby(["campus", "round"]):
    print(f"{c}{rd}:", round(pearson_r(g.temperature_c, g.pressure_hpa), 3))
corr = minutes[CHANNELS].corr()
plt.figure(figsize=(6.5, 5.2))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(10), CHANNELS, rotation=60, ha="right")
plt.yticks(range(10), CHANNELS)
plt.colorbar(label="r"); plt.title("Correlation of the ten channels")
plt.show()

### 2.5 Dew point

$$\gamma = \ln\frac{\mathrm{RH}}{100} + \frac{a\,T}{b + T}, \qquad T_d = \frac{b\,\gamma}{a - \gamma}, \quad a = 17.625,\ b = 243.04\ ^\circ\mathrm{C}$$

In [ ]:
def dew_point(t, rh):
    """Dew point (deg C) from the Magnus form (Alduchov & Eskridge, 1996)."""
    a, b = 17.625, 243.04
    g = np.log(np.asarray(rh, dtype=float) / 100.0) + a * t / (b + t)
    return b * g / (a - g)


minutes["dew_point_c"] = dew_point(minutes.temperature_c, minutes.rel_humidity_pct)
display(minutes.pivot_table(index=["campus", "round"], columns="point_type",
                            values="dew_point_c", aggfunc="median").round(1))

### Exercise 2.B

What is the dew point of air at 20 °C and 50 % relative humidity? Use `dew_point`.

In [ ]:
answer = None   # your answer
check("Exercise 2.B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = dew_point(20.0, 50.0)
print(round(answer, 2))
check("Exercise 2.B", answer)

**Observe and reflect.** On which campus and round is the dew point highest, and what does that say about the air of that morning? Why does the green area sit slightly higher?

## 3. Discover: AI finds patterns

*Workshop site: [Part 3](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/part3.html)*

### 3.1 Standardize

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

In [ ]:
def standardize(X):
    mu, sd = X.mean(axis=0), X.std(axis=0, ddof=0)
    return (X - mu) / sd, mu, sd


CLUSTER_FEATURES = ['temp_anomaly', 'log10_lux', 'pm25_anomaly']
X = anom[CLUSTER_FEATURES].to_numpy()
Z, mu, sd = standardize(X)
print("μ =", mu.round(3), "  σ =", sd.round(3))

### 3.2 k-means with k = 4

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(Z)
order = np.argsort(km.cluster_centers_[:, 0])          # coolest group first
rename = {old: new + 1 for new, old in enumerate(order)}
anom["group"] = pd.Series(km.labels_).map(rename).to_numpy()
centers = pd.DataFrame(km.cluster_centers_[order] * sd + mu,
                       columns=CLUSTER_FEATURES, index=[1, 2, 3, 4]).round(2)
display(centers)
display(pd.crosstab(anom["group"], anom["point_type"])[TYPE_ORDER])

In [ ]:
if HAS_PLOTLY:
    fig = px.scatter_3d(anom, x="temp_anomaly", y="log10_lux", z="pm25_anomaly",
                        color=anom["group"].astype(str), opacity=0.75,
                        hover_data=["campus", "round", "point_type"],
                        title="k-means groups of the clean minutes")
    fig.update_traces(marker_size=3)
    fig.show()
else:
    ax = plt.figure(figsize=(6, 5)).add_subplot(projection="3d")
    ax.scatter(anom.temp_anomaly, anom.log10_lux, anom.pm25_anomaly,
               c=anom.group, s=8)
    ax.set_xlabel("ΔT"); ax.set_ylabel("log10 lux"); ax.set_zlabel("ΔPM2.5")
    plt.show()

### 3.3 How many groups?

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
    rows.append(dict(k=k, J=round(m.inertia_, 1),
                     silhouette=round(silhouette_score(Z, m.labels_), 3)))
scores = pd.DataFrame(rows)
display(scores)

### Exercise 3.A

Which k has the highest silhouette score in `scores`?

In [ ]:
answer = None   # your answer
check("Exercise 3.A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(scores.loc[scores.silhouette.idxmax(), "k"])
print(answer)
check("Exercise 3.A", answer)

### 3.4 Principal component analysis

In [ ]:
from sklearn.decomposition import PCA

feats = ["temp_anomaly", "rel_humidity_pct", "pm25_anomaly", "co2_anomaly",
         "log10_lux", "uva_wm2"]
P = anom[feats].copy()
# a missing minute is placed at the median
P["co2_anomaly"] = P["co2_anomaly"].fillna(0.0)
Zp, _, _ = standardize(P.to_numpy())
pca = PCA(n_components=3, random_state=RANDOM_STATE).fit(Zp)
print("share of variation:", pca.explained_variance_ratio_.round(3))
loadings = pd.DataFrame(pca.components_, columns=feats, index=["PC1", "PC2", "PC3"])
display(loadings.round(2))
S = pca.transform(Zp)
if HAS_PLOTLY:
    fig = px.scatter_3d(x=S[:, 0], y=S[:, 1], z=S[:, 2], color=anom["point_type"],
                        color_discrete_map=TYPE_COLOR,
                        labels={"x": "PC1", "y": "PC2", "z": "PC3"})
    fig.update_traces(marker_size=3)
    fig.show()

**Observe and reflect.** Which group mixes several point types, and what do those minutes have in common? Would you trust the same four groups on a cloudy day?

## 4. Explain: Why is it high here?

*Workshop site: [Part 4](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/part4.html)*

### 4.1 The model table

In [ ]:
def model_table(minutes, forms, target_minutes=None):
    m = minutes.merge(
        forms[["campus", "round", "point_id", "vehicles_15min", "people_15min",
               "green_cover_pct", "sun_condition"]],
        on=["campus", "round", "point_id"], how="left")
    m["vehicles_per_min"] = m["vehicles_15min"] / 15.0
    m["people_per_min"] = m["people_15min"] / 15.0
    m["sun_index"] = m["sun_condition"].map({"shade": 0.0, "partial": 0.5,
                                             "sun": 1.0})
    ref = (m[m["point_type"] == "reference"]
           .groupby(["campus", "round"])["pm25_ugm3"].median()
           .rename("background_pm25"))
    m = m.merge(ref, on=["campus", "round"], how="left")
    return m


MODEL_FEATURES = ['vehicles_per_min', 'people_per_min', 'green_cover_pct',
    'sun_index', 'rel_humidity_pct', 'temperature_c', 'hour',
    'background_pm25']
table = model_table(minutes, forms).dropna(subset=MODEL_FEATURES + ["pm25_ugm3"])
Xm = table[MODEL_FEATURES].to_numpy()
y = table["pm25_ugm3"].to_numpy()
groups = (table.campus + table.point_id).to_numpy()
print(len(y), "minutes from", len(set(groups)), "measurement points")
display(table[MODEL_FEATURES + ["pm25_ugm3"]].head(3))

### 4.2 Three scores

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_predict


def make_forest():
    return RandomForestRegressor(n_estimators=40, max_depth=6, min_samples_leaf=5,
                                 random_state=RANDOM_STATE, n_jobs=-1)


forest = make_forest().fit(Xm, y)
p_random = cross_val_predict(make_forest(), Xm, y,
                             cv=KFold(5, shuffle=True, random_state=RANDOM_STATE))
p_group = cross_val_predict(make_forest(), Xm, y, cv=LeaveOneGroupOut(),
                            groups=groups)
scores_cv = pd.Series({"own training minutes": r2_score(y, forest.predict(Xm)),
                       "random minutes (5-fold)": r2_score(y, p_random),
                       "leave one point out": r2_score(y, p_group)}).round(3)
display(scores_cv.to_frame("R²"))

### 4.3 Permutation importance

Train on two campuses, shuffle one input at a time on the third and measure the drop of R². The three results are averaged.

In [ ]:
from sklearn.inspection import permutation_importance


def campus_importance(tab, target="pm25_ugm3"):
    X_, y_ = tab[MODEL_FEATURES].to_numpy(), tab[target].to_numpy()
    res = []
    for c in sorted(tab.campus.unique()):
        train = (tab.campus != c).to_numpy()
        m = make_forest().fit(X_[train], y_[train])
        pi = permutation_importance(m, X_[~train], y_[~train], n_repeats=10,
                                    random_state=RANDOM_STATE)
        res.append(pi.importances_mean)
    return pd.Series(np.mean(res, axis=0), index=MODEL_FEATURES).sort_values()


imp = campus_importance(table)
imp.plot.barh(figsize=(6, 3.4), color="#A65400")
plt.xlabel("drop of R² when shuffled"); plt.show()

### Exercise 4.A

Which input has the largest importance? Write its name as text, for example `"hour"`. The reference solution then repeats the importance for the raw PM2.5 without the humidity correction (`minutes_raw_pm`), so the two columns can be compared.

In [ ]:
answer = None   # your answer
check("Exercise 4.A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = imp.idxmax()
print(answer)
check("Exercise 4.A", answer)
raw_table = model_table(minutes_raw_pm, forms)
raw_table = raw_table.dropna(subset=MODEL_FEATURES + ["pm25_ugm3"])
both = pd.DataFrame({"corrected": imp, "raw": campus_importance(raw_table)})
display(both.round(3))

### 4.4 What if?

Move the sliders. The bars show how much the prediction falls when one input goes back to its median.

In [ ]:
import ipywidgets as widgets

med = np.median(Xm, axis=0)
sliders = {f: widgets.FloatSlider(value=float(med[i]), min=float(Xm[:, i].min()),
                                  max=float(Xm[:, i].max()),
                                  step=float((Xm[:, i].max() - Xm[:, i].min()) / 50),
                                  description=f[:18], continuous_update=False)
           for i, f in enumerate(MODEL_FEATURES)}


def what_if(**values):
    x = np.array([values[f] for f in MODEL_FEATURES])
    p = forest.predict([x])[0]
    effect = {}
    for i, f in enumerate(MODEL_FEATURES):
        x2 = x.copy(); x2[i] = med[i]
        effect[f] = p - forest.predict([x2])[0]
    s = pd.Series(effect).sort_values()
    colors = ["#1F5F8B" if v < 0 else "#A65400" for v in s]
    s.plot.barh(figsize=(6, 3.2), color=colors)
    plt.title(f"predicted PM2.5: {p:.1f} µg/m³"); plt.show()


out = widgets.interactive_output(what_if, sliders)
display(widgets.VBox(list(sliders.values())), out)

### 4.5 Partial dependence

$$\hat{f}_S(x_S) = \frac{1}{n}\sum_{i=1}^{n} \hat{f}\bigl(x_S, \mathbf{x}_C^{(i)}\bigr)$$

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(12, 3.2))
PartialDependenceDisplay.from_estimator(
    forest, Xm, features=[0, 7, 1, 2], feature_names=MODEL_FEATURES, ax=ax,
    kind="both", subsample=60, random_state=RANDOM_STATE)
plt.tight_layout(); plt.show()

### 4.6 The model surface in 3D

In [ ]:
ix = MODEL_FEATURES.index("vehicles_per_min")
iy = MODEL_FEATURES.index("background_pm25")
gx = np.linspace(Xm[:, ix].min(), Xm[:, ix].max(), 30)
gy = np.linspace(Xm[:, iy].min(), Xm[:, iy].max(), 30)
grid = np.tile(med, (900, 1))
grid[:, ix] = np.repeat(gx, 30)
grid[:, iy] = np.tile(gy, 30)
Zs = forest.predict(grid).reshape(30, 30).T
if HAS_PLOTLY:
    fig = go.Figure(go.Surface(x=gx, y=gy, z=Zs, colorscale="Blues"))
    fig.update_layout(scene=dict(xaxis_title="vehicles_per_min",
                                 yaxis_title="background_pm25", zaxis_title="PM2.5"),
                      height=520,
                      title="Predicted PM2.5, other inputs at medians")
    fig.show()
else:
    ax = plt.figure(figsize=(7, 5)).add_subplot(projection="3d")
    GX, GY = np.meshgrid(gx, gy)
    ax.plot_surface(GX, GY, Zs, cmap="Blues"); plt.show()

**Observe and reflect.** The forest found the rules that generated the synthetic data. Which field observation would you need to believe the same importance on a real campus?

## 5. Share and turn into action

*Workshop site: [Part 5](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/part5.html)*

### 5.1 The fact table

In [ ]:
CAMPUS = "B"
facts = pd.concat([fair_table(visits, ch).assign(channel=ch)
                   for ch in ["pm25_ugm3", "temperature_c", "co2_ppm"]])
facts = facts[facts.campus == CAMPUS][["channel", "point_type", "round1", "round2",
                                       "two_round_mean", "minus_reference"]].round(1)
facts.to_csv(f"nokta_facts_{CAMPUS}.csv", index=False)
display(facts)

### 5.2 A safe prompt

In [ ]:
RULES = [
    "Use only the numbers in the table, and write 'not measured' for a missing value.",
    "Do not rank campuses, universities or cities.",
    "State that each value is the median of a 15-minute visit.",
    "Do not compare the values with 24-hour or annual guidelines.",
    "Keep observation, interpretation and advice in separate paragraphs.",
    "Mention low-cost sensors, humidity correction and offsets.",
    "Use cautious verbs for causes."]


def table_text(frame):
    try:
        return frame.to_markdown(index=False)          # needs the tabulate package
    except ImportError:
        return frame.to_string(index=False)


def make_prompt(facts, campus, audience="a campus environment report"):
    lines = ["Role: You help a student team report campus measurements.",
             f"Task: Write {audience} about Campus {campus}.",
             "Data:", table_text(facts), "Rules:"]
    return "\n".join(lines + ["- " + r for r in RULES])


print(make_prompt(facts, CAMPUS))

### 5.3 The number auditor

Paste an AI-written text into `text`. The auditor lists every number and marks those that the fact table does not contain. A flagged number is not always wrong, for example a guideline value or a year, but every flagged number needs a source.

In [ ]:
def audit_numbers(text, facts, tol=0.051):
    """List every number of a text and whether the fact table supports it."""
    allowed = []
    for v in np.ravel(facts.select_dtypes("number").to_numpy()):
        if np.isfinite(v):
            allowed.append(float(v))
    rows = []
    for m in NUMBER.finditer(text):
        val = float(m.group().replace(",", "."))
        hit = any(abs(val - a) <= tol for a in allowed)
        rows.append(dict(number=m.group(), supported=hit,
                         context=text[max(0, m.start() - 30):m.end() + 30]))
    return pd.DataFrame(rows)


NUMBER = re.compile(r"(?<![\w.,])-?\d+(?:[.,]\d+)?")
text = ('On Campus B the road had the highest PM2.5, with a two-round mean of '
        '21.7 µg/m³, 7.8 µg/m³ above the reference. The green area was the '
        'cleanest point at 6.1 µg/m³. Near the road PM2.5 exceeded the WHO '
        '24-hour guideline of 15 µg/m³. A 2024 report confirms this. The '
        'sunny square was 3.6 °C warmer.')
audit = audit_numbers(text, facts)
display(audit)
print("numbers without support in the table:",
      int((~audit.supported).sum()))

### Exercise 5.A

How many numbers of `text` does the auditor flag as unsupported?

In [ ]:
answer = None   # your answer
check("Exercise 5.A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((~audit.supported).sum())
print(answer)
check("Exercise 5.A", answer)

### 5.4 The solution card

In [ ]:
def solution_card(table, problem, evidence, solution, feasibility, youth):
    parts = [("Problem", problem), ("Evidence", evidence), ("Solution", solution),
             ("Feasibility", feasibility), ("Role of young people", youth)]
    return f"### {table}\n\n" + "\n\n".join(f"**{k}.** {v}" for k, v in parts)


row = facts[(facts.channel == "temperature_c") & (facts.point_type == "sunny_hard")]
card = solution_card(
    "Microclimate and green campus",
    "The concrete square heats up in the sun.",
    f"Two-round mean {row.minus_reference.iloc[0]:+.1f} °C"
    " against the reference point.",
    "Shade sails over the benches and a row of trees.",
    "Low cost for the sails, while trees need one season.",
    "Students measure again after one year with the same protocol.")
display(Markdown(card))

## Application challenges

| Field | Challenge |
|---|---|
| Engineering | Design a test that separates real heat at the sunny square from sunshine on the sensor housing. |
| Health sciences | Estimate how long a student waits at the bus stop each day and what the PM2.5 difference means for that time. |
| Social sciences | Ask ten students where on campus they feel the air is worst and compare their answers with the fair table. |
| Communication | Turn one result into a figure for the congress that shows its averaging time and its uncertainty. |
| Economics and management | Compare the cost per degree of shade sails and trees for the sunny square over ten years. |
| Education | Plan a 40-minute lesson in which pupils repeat the two-round design with a thermometer. |

## After the notebook

Open the [debrief of the NOKTA Lab](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/lab.html#debrief) for a field report of the session. For the analysis days of 21 to 24 November, set `DATA_SOURCE = "own"` in Section 0 and run the notebook on your own NOKTA files.

**References**

1. Leys, C., Ley, C., Klein, O., Bernard, P., & Licata, L. (2013). Detecting outliers: Do not use standard deviation around the mean, use absolute deviation around the median. *Journal of Experimental Social Psychology, 49*(4), 764–766. https://doi.org/10.1016/j.jesp.2013.03.013
2. Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). Isolation forest. In *Proceedings of the 8th IEEE International Conference on Data Mining (ICDM 2008)* (pp. 413–422). https://doi.org/10.1109/ICDM.2008.17
3. Crilley, L. R., Shaw, M., Pound, R., Kramer, L. J., Price, R., Young, S., Lewis, A. C., & Pope, F. D. (2018). Evaluation of a low-cost optical particle counter (Alphasense OPC-N2) for ambient air monitoring. *Atmospheric Measurement Techniques, 11*(2), 709–720. https://doi.org/10.5194/amt-11-709-2018
4. Alduchov, O. A., & Eskridge, R. E. (1996). Improved Magnus form approximation of saturation vapor pressure. *Journal of Applied Meteorology, 35*(4), 601–609. https://doi.org/10.1175/1520-0450(1996)035<0601:IMFAOS>2.0.CO;2
5. Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory, 28*(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489
6. Jolliffe, I. T., & Cadima, J. (2016). Principal component analysis: A review and recent developments. *Philosophical Transactions of the Royal Society A, 374*(2065), 20150202. https://doi.org/10.1098/rsta.2015.0202
7. Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324
8. Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S., Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., & Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881
9. Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[utkukose.com](https://www.utkukose.com) · [ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [github.com/utkukose](https://github.com/utkukose)

<sub>NOKTA workshop · ÜNİDES 2026 · First version: November 2026 · Content CC BY 4.0, code MIT · The data are synthetic.</sub>

</div>